# 🎨🔊🎬 Actividad 15 — Modelos Multimodales: imagen, audio y vídeo

## ¿Qué vas a descubrir?

Un modelo de lenguaje solo entiende **vectores**. Entonces, ¿cómo "ve" una foto o "escucha" un audio?
La respuesta es que cada modalidad tiene su propio **tokenizador**, que la trocea en fragmentos
y convierte cada fragmento en un vector, igual que se hace con las palabras:

```
  TEXTO   "el gato duerme"   → tokens: [el] [gato] [duerme]          → 1 vector por token
  IMAGEN  🖼️ 224×224 px       → parches de 32×32 px (7×7 = 49)        → 1 vector por parche
  AUDIO   🔊 30 s             → espectrograma → tramos de 20 ms (1500) → 1 vector por tramo
  VÍDEO   🎬 N fotogramas     → N imágenes → N × (parches)            → muchísimos vectores
                                         │
                                         ▼
                        Todos viven en el mismo "idioma" de vectores
                                         ▼
                                       LLM
```

## Mapa de la actividad

| Parte | Tema |
|-------|------|
| 1 | Preparar Colab: Ollama + `gemma3:4b` y comprobar las librerías que ya trae Colab |
| 2 | **Imagen I** — el tokenizador visual: parches, vectores y posición (CLIP ViT) |
| 3 | **Imagen II** — imagen + texto: CLIP y un LLM multimodal (`gemma3:4b`) |
| 4 | **Audio** — espectrograma, tokens de audio y transcripción con Whisper |
| 5 | **Vídeo** — fotogramas, coste en tokens y fotogramas clave |
| 6 | Interfaz Gradio con las tres modalidades |
| 7 | Reflexión |
| 8 | Retos opcionales |
| 9 | Publicación con Ngrok + limpieza + rúbrica |

## Datos de ejemplo (públicos y sin subir nada)

| Modalidad | Ejemplo | Viene de |
|-----------|---------|----------|
| Imagen | Astronauta, gata *Chelsea*, taza de café | `skimage.data` (scikit-image, ya instalado en Colab) |
| Audio | Fragmento de audiolibro en inglés (LibriSpeech) | `librosa.ex('libri1')` (librosa, ya instalado en Colab) |
| Vídeo | Cacatúa (`cockatoo.mp4`) | Vídeo de ejemplo oficial de `imageio` (ya instalado en Colab) |

**⏱️ Duración:** 100–130 minutos · **GPU:** T4 (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*)

> 💡 **Recuerda:** tú solo ejecutas las celdas de arriba a abajo. Los comentarios del código explican todo.

## ⚙️ PARTE 1 — Preparar Colab

Casi todo lo que necesitamos **ya viene instalado en Colab**:

| Librería | ¿Para qué la usamos? |
|----------|----------------------|
| `transformers` + `torch` | Cargar CLIP (visión) y Whisper (audio) desde HuggingFace |
| `scikit-image` | Imágenes de ejemplo |
| `librosa` | Audio de ejemplo y su representación visual |
| `opencv` (`cv2`) + `imageio` | Leer el vídeo de ejemplo fotograma a fotograma |
| `matplotlib` | Dibujar parches, espectrogramas y mapas de calor |

Lo único que instalamos aparte es lo estándar del curso: **Ollama** (servidor del LLM multimodal),
**Gradio** (interfaz) y **pyngrok** (publicación).

```
  Tu notebook
    ├── transformers ─► CLIP ViT-B/32   (ver CÓMO se trocea una imagen)
    ├── transformers ─► Whisper small   (audio → texto)
    └── HTTP ─────────► Ollama ─► gemma3:4b  (LLM que entiende texto + imágenes)
```

In [ ]:
# ============================================================
# CELDA 1.1 — Comprobar que tenemos GPU
# ============================================================
# nvidia-smi muestra la tarjeta gráfica disponible. CLIP, Whisper y gemma3 van mucho
# más rápido en GPU (en CPU la actividad sería muy lenta).

!nvidia-smi --query-gpu=name,memory.total --format=csv

# Si ves "Tesla T4", perfecto. Si ves un error:
# Entorno de ejecución → Cambiar tipo de entorno → T4 GPU, y vuelve a empezar.
print('✅ Revisa arriba que aparezca la GPU (Tesla T4)')

In [ ]:
# ============================================================
# CELDA 1.2 — Verificar las librerías que ya trae Colab
# ============================================================
# No instalamos nada: solo importamos y mostramos versiones para confirmar que existen.
import torch, transformers, skimage, librosa, cv2, imageio, matplotlib

for nombre, modulo in [('torch', torch), ('transformers', transformers), ('scikit-image', skimage),
                       ('librosa', librosa), ('opencv', cv2), ('imageio', imageio),
                       ('matplotlib', matplotlib)]:
    print(f'   {nombre:<13} {modulo.__version__}')

# torch.cuda.is_available() es True si PyTorch puede usar la GPU
DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Librerías disponibles · PyTorch usará: {DISPOSITIVO.upper()}')

In [ ]:
# ============================================================
# CELDA 1.3 — Instalar Ollama
# ============================================================
# zstd es un descompresor que el instalador de Ollama necesita en Colab.
!apt-get install -y -q zstd > /dev/null

# Script oficial: descarga Ollama y lo deja disponible como comando de terminal.
!curl -fsSL https://ollama.com/install.sh | sh

print('✅ Ollama instalado')

In [ ]:
# ============================================================
# CELDA 1.4 — Instalar Gradio y pyngrok
# ============================================================
# gradio>=5,<6 → interfaz web (fijamos la versión para que el notebook no cambie de comportamiento)
# pyngrok      → publicaremos la app al final
!pip install -q 'gradio>=5,<6' pyngrok

print('✅ Gradio y pyngrok instalados')

In [ ]:
# ============================================================
# CELDA 1.5 — Arrancar el servidor de Ollama en segundo plano
# ============================================================
import subprocess, time, requests

# subprocess.Popen lanza el proceso en paralelo (no bloquea el notebook).
# DEVNULL descarta los mensajes del servidor para no saturar la salida.
proceso_ollama = subprocess.Popen(['ollama', 'serve'],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

OLLAMA_URL = 'http://localhost:11434'   # Puerto por defecto de Ollama

# Preguntamos cada segundo hasta que responda (máx. 30 s)
for intento in range(30):
    try:
        if requests.get(OLLAMA_URL, timeout=2).status_code == 200:
            print(f'✅ Servidor Ollama activo (tardó ~{intento} s)')
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    # El else de un for solo se ejecuta si el bucle terminó SIN break
    print('❌ Ollama no arrancó. Reinicia la sesión y ejecuta desde la celda 1.3.')

In [ ]:
# ============================================================
# CELDA 1.6 — Descargar el LLM multimodal gemma3:4b
# ============================================================
# Gemma 3 (Google) de 4 mil millones de parámetros, cuantizado (~3.3 GB).
# Lleva incorporado un codificador de visión (SigLIP): acepta texto E imágenes en el mismo mensaje.
# Tarda 2-4 minutos en descargarse.

MODELO_VLM = 'gemma3:4b'   # VLM = Vision-Language Model

resultado = subprocess.run(['ollama', 'pull', MODELO_VLM], capture_output=True, text=True)

if resultado.returncode == 0:   # 0 = terminó sin errores
    print(f'✅ {MODELO_VLM} descargado')
else:
    print(f'❌ Error: {resultado.stderr[-300:]}')
    print('   Vuelve a ejecutar esta celda; las descargas grandes a veces se cortan.')

In [ ]:
# ============================================================
# CELDA 1.7 — Importaciones de uso general
# ============================================================
import numpy as np                  # Las imágenes, audios y vídeos son, al final, matrices de números
import matplotlib.pyplot as plt     # Para dibujar
from PIL import Image               # Para convertir y redimensionar imágenes
from IPython.display import Audio, display   # Reproductor de audio dentro del notebook

plt.rcParams['figure.dpi'] = 100    # Resolución de las figuras

print('✅ PARTE 1 completa')

## 🖼️ PARTE 2 — Imagen I: el tokenizador visual

Un **Vision Transformer (ViT)** —el "ojo" de casi todos los LLM multimodales— no mira la imagen entera
de golpe. La procesa en 4 pasos, y aquí vamos a hacer **cada uno a mano** con **CLIP ViT-B/32**:

```
  ① Preprocesar           ② Trocear en parches          ③ Parche → vector              ④ + posición
  ┌──────────┐            ┌──┬──┬──┬──┬──┬──┬──┐        parche 32×32×3                vector + "estoy en
  │  foto    │  224×224   │ 0│ 1│ 2│ 3│ 4│ 5│ 6│        = 3072 números                 la fila 2, col 3"
  │ cualquier│ ─────────► ├──┼──┼──┼──┼──┼──┼──┤        ──── × matriz W ────►          ─────────────►
  │  tamaño  │ normalizar │ 7│ 8│ …│  │  │  │  │        768 números                   49 vectores + [CLS]
  └──────────┘            └──┴──┴──┴──┴──┴──┴──┘        (un "token visual")            = 50 tokens
                           7 × 7 = 49 parches
```

> 🔑 **Idea clave:** un parche de imagen para el modelo es exactamente lo mismo que una palabra:
> una posición en la secuencia con su vector. Por eso el mismo tipo de red (Transformer) sirve para ambos.

In [ ]:
# ============================================================
# CELDA 2.1 — Cargar las imágenes de ejemplo de scikit-image
# ============================================================
# skimage.data incluye imágenes de dominio público que vienen dentro de la librería
# (no se descargan de Internet). Cada imagen es un array de numpy: alto × ancho × 3 colores (RGB).
from skimage import data

IMAGENES = {
    'astronauta': data.astronaut(),   # Eileen Collins, astronauta de la NASA
    'gata':       data.chelsea(),     # La gata Chelsea
    'cafe':       data.coffee(),      # Una taza de café
}

fig, ejes = plt.subplots(1, 3, figsize=(12, 4))
for eje, (nombre, img) in zip(ejes, IMAGENES.items()):
    eje.imshow(img)
    eje.set_title(f'{nombre}  {img.shape}')   # shape = (alto, ancho, canales)
    eje.axis('off')
plt.show()
print('✅ 3 imágenes cargadas')

In [ ]:
# ============================================================
# CELDA 2.2 — Una imagen es una matriz de números
# ============================================================
# Antes de hablar de tokens, comprobemos qué es una imagen para el ordenador.

img = IMAGENES['gata']
print(f'Forma: {img.shape}  →  {img.shape[0]} filas × {img.shape[1]} columnas × 3 canales (R, G, B)')
print(f'Tipo de dato: {img.dtype}  →  enteros de 0 (oscuro) a 255 (máxima intensidad)')
print(f'Total de números: {img.size:,}')

# Un píxel concreto: [rojo, verde, azul]
print(f'Píxel en la fila 100, columna 200: {img[100, 200]}')
print('✅ Una foto no es más que una tabla enorme de números')

In [ ]:
# ============================================================
# CELDA 2.3 — Cargar CLIP ViT-B/32 (procesador + modelo)
# ============================================================
# CLIP (OpenAI) tiene dos partes: un codificador de imágenes (ViT) y uno de texto.
# "B/32" = tamaño Base con parches de 32×32 píxeles. Pesa ~600 MB.
# Usamos una versión con parches grandes a propósito: salen pocos parches (49) y se ven bien.
from transformers import CLIPModel, CLIPProcessor

NOMBRE_CLIP = 'openai/clip-vit-base-patch32'
procesador_clip = CLIPProcessor.from_pretrained(NOMBRE_CLIP)   # El "tokenizador" de imágenes (y de texto)
modelo_clip = CLIPModel.from_pretrained(NOMBRE_CLIP).to(DISPOSITIVO).eval()   # eval() = modo inferencia

cfg = modelo_clip.config.vision_config
print(f'Tamaño de entrada: {cfg.image_size}×{cfg.image_size} px')
print(f'Tamaño de parche:  {cfg.patch_size}×{cfg.patch_size} px')
print(f'Dimensión de cada vector: {cfg.hidden_size}')
print('✅ CLIP cargado')

In [ ]:
# ============================================================
# CELDA 2.4 — Paso ① Preprocesar: redimensionar, recortar y normalizar
# ============================================================
# Todas las imágenes deben tener el MISMO tamaño (224×224) porque el modelo
# tiene un número fijo de "casillas" de posición. El procesador:
#   1. Redimensiona el lado corto a 224 px
#   2. Recorta el centro a 224×224 (¡lo que quede fuera se pierde!)
#   3. Normaliza: (valor/255 - media) / desviación, por canal. Así los números quedan cerca de 0.

entrada = procesador_clip(images=img, return_tensors='pt')   # 'pt' = tensores de PyTorch
pixeles = entrada['pixel_values']                            # Forma: (lote, canales, alto, ancho)

print(f'Antes:   {img.shape}  (alto, ancho, canales)  valores 0..255')
print(f'Después: {tuple(pixeles.shape)}  (lote, canales, alto, ancho)  '
      f'valores {pixeles.min():.2f}..{pixeles.max():.2f}')

# Para poder VER la imagen preprocesada deshacemos la normalización
MEDIA = np.array(procesador_clip.image_processor.image_mean)
DESV = np.array(procesador_clip.image_processor.image_std)

def a_imagen_visible(pixel_values):
    """(1, 3, 224, 224) normalizado → (224, 224, 3) en 0..1 para matplotlib."""
    x = pixel_values[0].permute(1, 2, 0).cpu().numpy()   # permute reordena ejes: canales al final
    return np.clip(x * DESV + MEDIA, 0, 1)               # Deshace la normalización

img_224 = a_imagen_visible(pixeles)
plt.imshow(img_224); plt.title('Lo que realmente recibe el modelo (224×224)'); plt.axis('off'); plt.show()
print('✅ Imagen preprocesada')

In [ ]:
# ============================================================
# CELDA 2.5 — Paso ② Trocear la imagen en parches
# ============================================================
# Dividimos la imagen de 224×224 en una cuadrícula de parches de 32×32 → 7 filas × 7 columnas.
# Cada parche será UN token visual.

P = cfg.patch_size                 # 32
N_LADO = cfg.image_size // P       # 224 // 32 = 7 parches por lado
N_PARCHES = N_LADO * N_LADO        # 49

def dibujar_cuadricula(img_vis, n_lado, titulo=''):
    """Dibuja la imagen con la cuadrícula de parches y el número de cada parche."""
    tam = img_vis.shape[0] // n_lado
    fig, eje = plt.subplots(figsize=(6, 6))
    eje.imshow(img_vis)
    for k in range(1, n_lado):                      # Líneas de la cuadrícula
        eje.axhline(k * tam, color='yellow', lw=1)
        eje.axvline(k * tam, color='yellow', lw=1)
    for f in range(n_lado):                         # Número de cada parche en su esquina
        for c in range(n_lado):
            eje.text(c * tam + 2, f * tam + 9, str(f * n_lado + c), color='yellow', fontsize=8,
                     bbox=dict(facecolor='black', alpha=0.5, pad=1, lw=0))
    eje.set_title(titulo); eje.axis('off'); plt.show()

dibujar_cuadricula(img_224, N_LADO, f'{N_LADO}×{N_LADO} = {N_PARCHES} parches de {P}×{P} px')
print(f'✅ La imagen se convierte en una secuencia de {N_PARCHES} parches, leídos de izquierda a derecha '
      'y de arriba abajo (como un texto)')

In [ ]:
# ============================================================
# CELDA 2.6 — Ver los parches como una "frase" de tokens
# ============================================================
# Extraemos los 49 parches y los mostramos en fila, en el ORDEN en que entran al modelo.
# Es la misma idea que partir una frase en tokens: [el] [gato] [duerme] ...

def extraer_parches(pixel_values, p):
    """(1, 3, 224, 224) → (49, 3, 32, 32): lista de parches en orden de lectura."""
    x = pixel_values[0]                                  # (3, 224, 224)
    # unfold "desliza" una ventana de tamaño p con paso p en el eje 1 (alto) y en el 2 (ancho)
    x = x.unfold(1, p, p).unfold(2, p, p)                # (3, 7, 7, 32, 32)
    return x.permute(1, 2, 0, 3, 4).reshape(-1, 3, p, p) # (49, 3, 32, 32)

parches = extraer_parches(pixeles, P)
print(f'Forma de los parches: {tuple(parches.shape)}  → (nº parches, canales, alto, ancho)')

fig, ejes = plt.subplots(3, 17, figsize=(17, 3.4))
for i, eje in enumerate(ejes.flat):
    eje.axis('off')
    if i < N_PARCHES:
        eje.imshow(np.clip(parches[i].permute(1, 2, 0).numpy() * DESV + MEDIA, 0, 1))
        eje.set_title(str(i), fontsize=7)
plt.suptitle('La imagen como secuencia de 49 "tokens visuales"'); plt.show()
print('✅ Parches extraídos')

In [ ]:
# ============================================================
# CELDA 2.7 — Paso ③ Un parche se convierte en vector (a mano)
# ============================================================
# Cada parche tiene 32 × 32 × 3 = 3072 números. El modelo lo "aplana" en una fila
# y lo multiplica por una matriz de pesos W (aprendida en el entrenamiento) de 768 × 3072.
# Resultado: un vector de 768 números = el embedding de ese token visual.
# Es exactamente lo que hace la tabla de embeddings con las palabras, pero con píxeles.

capa_parches = modelo_clip.vision_model.embeddings.patch_embedding   # Conv2d(3, 768, kernel=32, stride=32)
W = capa_parches.weight.detach().cpu()                               # Pesos: (768, 3, 32, 32)
print(f'Capa que convierte parches en vectores: {capa_parches}')

i_parche = 24                                    # El parche central de la cuadrícula 7×7
parche_plano = parches[i_parche].reshape(-1)     # (3072,)  → el parche como una fila de números
vector_manual = W.reshape(768, -1) @ parche_plano   # (768, 3072) @ (3072,) = (768,)

print(f'Parche {i_parche}: {tuple(parches[i_parche].shape)} → aplanado {tuple(parche_plano.shape)} '
      f'→ vector {tuple(vector_manual.shape)}')
print(f'Primeros 8 números del vector: {np.round(vector_manual[:8].numpy(), 3)}')
print('✅ 3072 píxeles → 768 números: así nace un token visual')

In [ ]:
# ============================================================
# CELDA 2.8 — Comprobar que el modelo hace lo mismo que nosotros
# ============================================================
# Una Conv2d con kernel=32 y stride=32 aplica la MISMA matriz W a cada parche sin solaparlos.
# Por eso los ViT la usan: es una forma rápida de hacer "aplanar + multiplicar" en los 49 a la vez.

with torch.no_grad():   # no_grad: no calculamos gradientes (solo inferencia) → menos memoria
    salida_conv = capa_parches(pixeles.to(DISPOSITIVO))   # (1, 768, 7, 7)

fila, col = divmod(i_parche, N_LADO)                      # Parche 24 → fila 3, columna 3
vector_modelo = salida_conv[0, :, fila, col].cpu()

# allclose compara dos tensores permitiendo diferencias mínimas de redondeo
print(f'Salida de la convolución: {tuple(salida_conv.shape)} → 768 valores para cada casilla 7×7')
print(f'¿Nuestro vector coincide con el del modelo? {torch.allclose(vector_manual, vector_modelo, atol=1e-3)}')
print('✅ La "magia" de convertir una imagen en tokens es una multiplicación de matrices')

In [ ]:
# ============================================================
# CELDA 2.9 — Paso ④ Añadir el token [CLS] y la posición
# ============================================================
# Problema: después del paso ③ el modelo no sabe DÓNDE estaba cada parche (arriba, abajo...).
# Solución: se suma a cada vector un "vector de posición" aprendido (uno por casilla).
# Además se añade al principio un token especial [CLS] que acabará resumiendo toda la imagen.

emb = modelo_clip.vision_model.embeddings
with torch.no_grad():
    secuencia = emb(pixeles.to(DISPOSITIVO))     # Hace ②+③+④ de una vez: (1, 50, 768)
posiciones = emb.position_embedding.weight.detach().cpu()   # (50, 768): un vector por posición

print(f'Secuencia de entrada al Transformer: {tuple(secuencia.shape)}')
print(f'   = 1 token [CLS] + {N_PARCHES} parches = {secuencia.shape[1]} tokens de {secuencia.shape[2]} dimensiones')

# Similitud entre el vector de posición de un parche y el de todos los demás:
# los parches vecinos tienen posiciones parecidas → el modelo "sabe" quién está al lado de quién.
pos = torch.nn.functional.normalize(posiciones[1:], dim=1)   # Quitamos [CLS] y normalizamos
sim_pos = (pos @ pos[i_parche]).reshape(N_LADO, N_LADO)

plt.imshow(sim_pos, cmap='viridis'); plt.colorbar(label='similitud coseno')
plt.title(f'Parecido del vector de POSICIÓN del parche {i_parche} con el resto'); plt.show()
print('✅ Las posiciones cercanas tienen vectores parecidos: el modelo conserva la geometría')

In [ ]:
# ============================================================
# CELDA 2.10 — Pasar la secuencia por el Transformer y ver qué aprendió
# ============================================================
# El Transformer mezcla la información de todos los parches (atención).
# Al final, cada parche tiene un vector que "entiende" su contexto.
# Comparamos un parche con todos: los que muestran lo mismo (p. ej. pelaje) deberían parecerse.

with torch.no_grad():
    salida_vit = modelo_clip.vision_model(pixel_values=pixeles.to(DISPOSITIVO))
ocultos = salida_vit.last_hidden_state[0, 1:].cpu()          # (49, 768): quitamos [CLS]
ocultos = torch.nn.functional.normalize(ocultos, dim=1)

i_ref = 24                                                   # Parche de referencia
sim = (ocultos @ ocultos[i_ref]).reshape(N_LADO, N_LADO).numpy()

fig, ejes = plt.subplots(1, 2, figsize=(10, 4.5))
ejes[0].imshow(img_224); ejes[0].set_title(f'Imagen (parche {i_ref} marcado)'); ejes[0].axis('off')
f, c = divmod(i_ref, N_LADO)
ejes[0].add_patch(plt.Rectangle((c * P, f * P), P, P, fill=False, ec='red', lw=2))
im = ejes[1].imshow(sim, cmap='magma'); ejes[1].set_title('Parecido con los demás parches (salida del ViT)')
plt.colorbar(im, ax=ejes[1]); plt.show()
print('✅ Las zonas claras son parches que el modelo considera "del mismo tipo" que el marcado')

## 🔗 PARTE 3 — Imagen II: imagen + texto en el mismo espacio

### CLIP: imágenes y frases en el mismo "mapa"

CLIP se entrenó con 400 millones de pares (imagen, pie de foto). Aprendió a poner el vector de una foto
**cerca** del vector de la frase que la describe. Eso permite clasificar sin entrenar (*zero-shot*).

### ¿Y un LLM que conversa sobre imágenes?

Los LLM multimodales (LLaVA, Gemma 3, GPT-4o, Claude...) usan esta misma pieza:

```
  imagen ─► [codificador ViT] ─► N vectores ─► [proyector] ─► N "tokens de imagen" ┐
                                                                                    ├─► LLM ─► texto
  "¿Qué ves?" ─────────────────► [tokenizador de texto] ─► tokens de texto ─────────┘
```

| Modelo | Codificador | Resolución | Parche | Tokens por imagen |
|--------|-------------|-----------:|-------:|------------------:|
| CLIP ViT-B/32 (Parte 2) | ViT | 224 px | 32 px | 49 (+1 CLS) |
| LLaVA 1.5 | CLIP ViT-L/14 | 336 px | 14 px | 576 |
| **Gemma 3** | SigLIP | 896 px | 14 px | 4096 → agrupados a **256** |

Vamos a **medir** cuántos tokens ocupa una imagen en `gemma3:4b`.

In [ ]:
# ============================================================
# CELDA 3.1 — CLIP: clasificar una imagen comparándola con frases
# ============================================================
# Codificamos la imagen y varias frases, y medimos qué frase está más cerca.

frases = ['una foto de un gato', 'una foto de un perro', 'una foto de una astronauta',
          'una foto de una taza de café', 'una foto de un coche']

def clip_clasificar(imagen, textos):
    entrada = procesador_clip(text=textos, images=imagen, return_tensors='pt', padding=True).to(DISPOSITIVO)
    with torch.no_grad():
        salida = modelo_clip(**entrada)
    # logits_per_image = similitud imagen-frase escalada; softmax la convierte en probabilidades
    return salida.logits_per_image.softmax(dim=1)[0].cpu().numpy()

for nombre, imagen in IMAGENES.items():
    probs = clip_clasificar(imagen, frases)
    mejor = int(probs.argmax())
    print(f'{nombre:<11} → "{frases[mejor]}" ({probs[mejor]:.0%})')

print('✅ CLIP relaciona imágenes y textos sin haber sido entrenado para estas etiquetas concretas')

In [ ]:
# ============================================================
# CELDA 3.2 — Función para enviar imágenes a gemma3 vía Ollama
# ============================================================
# Ollama recibe las imágenes como texto en base64 (una forma de escribir bytes con letras).
import base64, io

def imagen_a_base64(imagen_np):
    """Array numpy (alto, ancho, 3) → PNG codificado en base64."""
    buffer = io.BytesIO()                                   # "Archivo" en memoria
    Image.fromarray(imagen_np.astype(np.uint8)).save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode()

def preguntar_vlm(pregunta, imagenes=None, temperatura=0.2):
    """Envía texto (+ imágenes opcionales) a gemma3. Devuelve (respuesta, tokens_de_entrada)."""
    mensaje = {'role': 'user', 'content': pregunta}
    if imagenes:
        mensaje['images'] = [imagen_a_base64(im) for im in imagenes]   # Lista de imágenes en base64
    r = requests.post(f'{OLLAMA_URL}/api/chat',
                      json={'model': MODELO_VLM, 'messages': [mensaje], 'stream': False,
                            'options': {'temperature': temperatura}},
                      timeout=600)
    r.raise_for_status()
    datos = r.json()
    # prompt_eval_count = cuántos tokens procesó el modelo como ENTRADA (texto + imagen)
    return datos['message']['content'].strip(), datos.get('prompt_eval_count', 0)

print('✅ Función preguntar_vlm definida')

In [ ]:
# ============================================================
# CELDA 3.3 — Probar gemma3 describiendo una imagen
# ============================================================
# La primera llamada tarda más (Ollama carga el modelo en la GPU).

inicio = time.time()
respuesta, _ = preguntar_vlm('Describe esta imagen en español en 3 frases.', [IMAGENES['astronauta']])
print(respuesta)
print(f'\n✅ gemma3 ve la imagen ({time.time() - inicio:.1f} s)')

In [ ]:
# ============================================================
# CELDA 3.4 — Medir cuántos tokens ocupa una imagen dentro del LLM
# ============================================================
# Hacemos la MISMA pregunta con y sin imagen. La diferencia de tokens de entrada
# es lo que "cuesta" la imagen en la ventana de contexto del LLM.
# (Ollama reutiliza parte del prompt entre llamadas, así que el número es aproximado.)

pregunta = '¿Qué color predomina? Responde con una palabra.'
_, tokens_sin = preguntar_vlm(pregunta)
_, tokens_con = preguntar_vlm(pregunta, [IMAGENES['cafe']])

print(f'Tokens de entrada SIN imagen: {tokens_sin}')
print(f'Tokens de entrada CON imagen: {tokens_con}')
print(f'➡️  La imagen ocupa ≈ {tokens_con - tokens_sin} tokens')

# Comparación: ¿cuántas palabras de texto equivalen a esos tokens?
print(f'   Es como añadir ≈ {int((tokens_con - tokens_sin) * 0.75)} palabras de texto al prompt')
print('✅ En Gemma 3 cada imagen se resume en ~256 tokens, sea del tamaño que sea')

In [ ]:
# ============================================================
# CELDA 3.5 — Una pregunta que requiere "mirar" de verdad
# ============================================================
# Preguntamos algo concreto para comprobar que el modelo usa la imagen y no inventa.

respuesta, _ = preguntar_vlm('¿Qué animal aparece y de qué color son sus ojos? Responde en español.',
                             [IMAGENES['gata']])
print(respuesta)
print('\n✅ PARTE 3 completa. Compara con la foto de la celda 2.1: ¿acertó?')

## 🔊 PARTE 4 — Audio: del sonido a los tokens con Whisper

Un audio es una lista de números (la presión del aire medida 16 000 veces por segundo).
Whisper no lo procesa así: primero lo convierte en una **imagen del sonido** (espectrograma)
y luego hace algo muy parecido a lo que vimos con los parches.

```
  Onda (16 000 números/segundo)
        │  ventanas de 25 ms cada 10 ms + filtro "mel" (como el oído humano)
        ▼
  Espectrograma log-mel: 80 bandas de frecuencia × 3000 columnas (= 30 s)
        │  2 convoluciones (la 2ª con paso 2) → como los parches de la imagen
        ▼
  1500 vectores de 768 números  (1 "token de audio" cada 20 ms)
        │  Encoder Transformer
        ▼
  Decoder: genera tokens de TEXTO  [<|en|>] [<|transcribe|>] [ He] [ was] ...
```

> 🔑 Whisper siempre trabaja en **ventanas de 30 segundos**. Los audios más largos se trocean;
> los más cortos se rellenan con silencio.

In [ ]:
# ============================================================
# CELDA 4.1 — Descargar el audio de ejemplo de librosa
# ============================================================
# librosa.ex() descarga (una sola vez) audios de ejemplo libres de derechos.
# 'libri1' es un fragmento del corpus LibriSpeech: audiolibros de dominio público en inglés.

ruta_audio = librosa.ex('libri1')

# sr=16000: lo leemos directamente a 16 kHz porque es la frecuencia con la que se entrenó Whisper.
audio, sr = librosa.load(ruta_audio, sr=16000)

print(f'Archivo: {ruta_audio}')
print(f'Muestras: {len(audio):,}  ·  Frecuencia: {sr} Hz  ·  Duración: {len(audio) / sr:.1f} s')
display(Audio(audio, rate=sr))   # Reproductor: dale al ▶ para escucharlo
print('✅ Audio cargado')

In [ ]:
# ============================================================
# CELDA 4.2 — Ver la onda: el audio "crudo"
# ============================================================
# Cada punto es la amplitud del sonido en un instante. Así no se distingue qué se dice.

t = np.arange(len(audio)) / sr   # Eje de tiempo en segundos
plt.figure(figsize=(12, 2.5))
plt.plot(t, audio, lw=0.4)
plt.xlabel('segundos'); plt.ylabel('amplitud'); plt.title('Forma de onda')
plt.show()
print(f'✅ {len(audio):,} números para {len(audio) / sr:.0f} segundos: demasiados para un Transformer')

In [ ]:
# ============================================================
# CELDA 4.3 — Cargar Whisper small (procesador + modelo)
# ============================================================
# Whisper (OpenAI) es un modelo de reconocimiento de voz multilingüe (99 idiomas).
# 'small' = 244 millones de parámetros (~1 GB), buen equilibrio calidad/velocidad en una T4.
# float16 = media precisión: la mitad de memoria y más rápido en GPU.
from transformers import WhisperProcessor, WhisperForConditionalGeneration

NOMBRE_WHISPER = 'openai/whisper-small'
TIPO = torch.float16 if DISPOSITIVO == 'cuda' else torch.float32   # En CPU float16 no conviene

procesador_whisper = WhisperProcessor.from_pretrained(NOMBRE_WHISPER)
modelo_whisper = WhisperForConditionalGeneration.from_pretrained(
    NOMBRE_WHISPER, dtype=TIPO).to(DISPOSITIVO).eval()

print(f'✅ Whisper cargado en {DISPOSITIVO.upper()} ({TIPO})')

In [ ]:
# ============================================================
# CELDA 4.4 — El "tokenizador" de audio: espectrograma log-mel
# ============================================================
# El procesador corta/rellena a 30 s y calcula el espectrograma: 80 bandas × 3000 columnas.
# Cada columna resume 10 ms de sonido. Es literalmente una imagen en escala de grises.

caracteristicas = procesador_whisper(audio, sampling_rate=16000, return_tensors='pt').input_features
print(f'Espectrograma: {tuple(caracteristicas.shape)}  → (lote, bandas de frecuencia, columnas de tiempo)')

plt.figure(figsize=(12, 3))
plt.imshow(caracteristicas[0], aspect='auto', origin='lower', cmap='magma')
plt.axvline(len(audio) / sr * 100, color='cyan', ls='--', label='fin del audio real (resto = relleno)')
plt.xlabel('columnas (1 = 10 ms)'); plt.ylabel('banda mel (graves → agudos)')
plt.title('Lo que "ve" Whisper: espectrograma log-mel de 30 s'); plt.legend(); plt.show()
print('✅ Las franjas claras son la voz: cada sílaba deja una "huella" de frecuencias')

In [ ]:
# ============================================================
# CELDA 4.5 — Del espectrograma a los tokens de audio (encoder)
# ============================================================
# El encoder aplica 2 convoluciones (la segunda con paso 2 → reduce 3000 columnas a 1500)
# y un Transformer. Resultado: 1500 vectores, uno cada 20 ms.
# Son el equivalente a los 49 vectores de parches de la imagen.

with torch.no_grad():
    tokens_audio = modelo_whisper.model.encoder(caracteristicas.to(DISPOSITIVO, TIPO)).last_hidden_state

print(f'Salida del encoder: {tuple(tokens_audio.shape)}  → (lote, tokens de audio, dimensiones)')
print(f'   30 s / {tokens_audio.shape[1]} tokens = {30000 / tokens_audio.shape[1]:.0f} ms por token')

# Mapa de calor de los primeros 64 valores de cada token: se ve cuándo hay voz y cuándo silencio
plt.figure(figsize=(12, 2.5))
plt.imshow(tokens_audio[0, :, :64].float().cpu().T, aspect='auto', cmap='coolwarm')
plt.xlabel('token de audio (1 = 20 ms)'); plt.ylabel('dimensión (64 de 768)')
plt.title('Vectores de audio a la salida del encoder'); plt.show()
print('✅ El audio ya es una secuencia de vectores, igual que una frase o una imagen')

In [ ]:
# ============================================================
# CELDA 4.6 — Transcribir y ver los tokens de texto que genera el decoder
# ============================================================
# generate() produce token a token. No le decimos el idioma: Whisper lo detecta solo
# y lo indica con un token especial como <|en|> o <|es|>.

with torch.no_grad():
    ids = modelo_whisper.generate(caracteristicas.to(DISPOSITIVO, TIPO), task='transcribe')

tokens = procesador_whisper.tokenizer.convert_ids_to_tokens(ids[0])   # ids → piezas de texto
print('Primeros 15 tokens generados:', tokens[:15])
texto = procesador_whisper.batch_decode(ids, skip_special_tokens=True)[0].strip()
print(f'\n📝 Transcripción:\n{texto}')
print('\n✅ Fíjate en los tokens especiales del principio: idioma detectado y tarea')

In [ ]:
# ============================================================
# CELDA 4.7 — Pipeline de transcripción con marcas de tiempo
# ============================================================
# pipeline() empaqueta todo lo anterior y además:
#   - chunk_length_s=30 trocea audios largos en ventanas de 30 s
#   - return_timestamps=True devuelve cuándo empieza y termina cada frase
from transformers import pipeline

transcriptor = pipeline('automatic-speech-recognition',
                        model=modelo_whisper,
                        tokenizer=procesador_whisper.tokenizer,
                        feature_extractor=procesador_whisper.feature_extractor,
                        chunk_length_s=30,
                        dtype=TIPO,               # Media precisión también para las entradas
                        device=DISPOSITIVO)

def transcribir(ruta_o_array):
    """Devuelve (texto, lista de fragmentos con tiempos)."""
    r = transcriptor(ruta_o_array, return_timestamps=True)
    return r['text'].strip(), r.get('chunks', [])

texto_audio, trozos = transcribir(ruta_audio)
for tr in trozos:
    ini, fin = tr['timestamp']
    print(f'[{ini:5.1f}s → {fin if fin is not None else 0:5.1f}s] {tr["text"].strip()}')
print('✅ Transcripción con tiempos lista')

In [ ]:
# ============================================================
# CELDA 4.8 — Encadenar modalidades: audio → texto → LLM
# ============================================================
# Muchos asistentes de voz funcionan así: Whisper convierte la voz en texto
# y un LLM trabaja sobre ese texto (aquí, resumir y traducir al español).

def resumir_transcripcion(texto):
    respuesta, _ = preguntar_vlm(
        'Este es un audio transcrito. Resúmelo en español en 2 frases y di de qué trata:\n\n' + texto)
    return respuesta

print(resumir_transcripcion(texto_audio))
print('\n✅ PARTE 4 completa: audio → tokens de audio → texto → LLM')

## 🎬 PARTE 5 — Vídeo: una secuencia de imágenes (y su coste)

Para un modelo, un vídeo es una **lista de fotogramas** (imágenes) en el tiempo.
El problema es el coste: si cada imagen son ~256 tokens...

```
  10 s × 25 fotogramas/s = 250 imágenes × 256 tokens = 64 000 tokens   😱
```

Por eso los sistemas reales **muestrean** pocos fotogramas y eligen los más distintos (**fotogramas clave**):

```
  Vídeo:      🖼️🖼️🖼️🖼️🖼️🖼️🖼️🖼️🖼️🖼️🖼️🖼️  (cientos de fotogramas)
                   │  muestreo uniforme (p. ej. 12)
                   ▼
              🖼️ 🖼️ 🖼️ 🖼️ 🖼️ 🖼️ 🖼️ 🖼️ 🖼️ 🖼️ 🖼️ 🖼️
                   │  CLIP: ¿se parece mucho al anterior elegido? → descartar
                   ▼
              🖼️       🖼️          🖼️       (fotogramas clave)
                   │  gemma3 describe cada uno + LLM resume la secuencia
                   ▼
              "Una cacatúa ... luego ... al final ..."
```

In [ ]:
# ============================================================
# CELDA 5.1 — Descargar el vídeo de ejemplo de imageio
# ============================================================
# 'cockatoo.mp4' es uno de los archivos de ejemplo oficiales de la librería imageio,
# alojado en su repositorio público imageio-binaries.
import urllib.request

URL_VIDEO = 'https://github.com/imageio/imageio-binaries/raw/master/images/cockatoo.mp4'
ruta_video = 'cockatoo.mp4'
urllib.request.urlretrieve(URL_VIDEO, ruta_video)   # Descarga el archivo a la carpeta actual

import os
print(f'✅ Vídeo descargado: {ruta_video} ({os.path.getsize(ruta_video) / 1e6:.1f} MB)')

In [ ]:
# ============================================================
# CELDA 5.2 — Leer las propiedades del vídeo con OpenCV
# ============================================================
# cv2.VideoCapture abre el vídeo; con get() consultamos sus metadatos.

cap = cv2.VideoCapture(ruta_video)
FPS = cap.get(cv2.CAP_PROP_FPS)                      # Fotogramas por segundo
N_FOTOGRAMAS = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ANCHO = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
ALTO = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()                                        # Cerramos el archivo

DURACION = N_FOTOGRAMAS / FPS
print(f'Resolución: {ANCHO}×{ALTO} · {FPS:.0f} fps · {N_FOTOGRAMAS} fotogramas · {DURACION:.1f} s')

# Coste si enviáramos TODOS los fotogramas a gemma3 (~256 tokens cada uno)
print(f'Si enviáramos todos: {N_FOTOGRAMAS} × 256 = {N_FOTOGRAMAS * 256:,} tokens  😱')
print('✅ Metadatos leídos')

In [ ]:
# ============================================================
# CELDA 5.3 — Definir el muestreo uniforme de fotogramas
# ============================================================

def extraer_fotogramas(ruta, n=12):
    """Devuelve n fotogramas repartidos uniformemente y su instante en segundos."""
    cap = cv2.VideoCapture(ruta)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25            # Si el vídeo no informa fps, suponemos 25
    fotogramas, tiempos = [], []
    # linspace reparte n posiciones equidistantes entre el primer y el último fotograma
    for idx in np.linspace(0, total - 1, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))   # Saltamos directamente a ese fotograma
        ok, fotograma = cap.read()
        if ok:
            # OpenCV guarda los colores como BGR (azul-verde-rojo); los pasamos a RGB
            fotogramas.append(cv2.cvtColor(fotograma, cv2.COLOR_BGR2RGB))
            tiempos.append(idx / fps)
    cap.release()
    return fotogramas, tiempos

print('✅ Función extraer_fotogramas definida')

In [ ]:
# ============================================================
# CELDA 5.4 — Extraer y ver 12 fotogramas
# ============================================================

fotogramas, tiempos = extraer_fotogramas(ruta_video, n=12)

fig, ejes = plt.subplots(2, 6, figsize=(16, 5))
for eje, fg, t in zip(ejes.flat, fotogramas, tiempos):
    eje.imshow(fg); eje.set_title(f'{t:.1f} s'); eje.axis('off')
plt.suptitle('12 fotogramas muestreados uniformemente'); plt.show()
print(f'✅ {len(fotogramas)} fotogramas → {len(fotogramas) * 256:,} tokens aprox. (vs {N_FOTOGRAMAS * 256:,})')

In [ ]:
# ============================================================
# CELDA 5.5 — Medir cuánto se parecen los fotogramas con CLIP
# ============================================================
# Reutilizamos el codificador de imágenes de CLIP: cada fotograma → 1 vector (512 números).
# Si dos fotogramas consecutivos tienen vectores muy parecidos, uno de ellos sobra.

def vectores_clip(imagenes):
    entrada = procesador_clip(images=imagenes, return_tensors='pt').to(DISPOSITIVO)
    with torch.no_grad():
        v = modelo_clip.get_image_features(**entrada)        # (n, 512): un vector por imagen
    # En transformers 5.x devuelve un objeto; el vector proyectado está en .pooler_output
    if not torch.is_tensor(v):
        v = v.pooler_output
    return torch.nn.functional.normalize(v, dim=1).cpu()     # Normalizados → producto punto = coseno

V = vectores_clip(fotogramas)
sim_consecutivos = (V[1:] * V[:-1]).sum(dim=1).numpy()       # Coseno entre cada fotograma y el anterior

plt.figure(figsize=(10, 2.5))
plt.plot(tiempos[1:], sim_consecutivos, 'o-')
plt.ylabel('similitud con el anterior'); plt.xlabel('segundos'); plt.ylim(0.5, 1.01)
plt.title('¿Cuánto cambia la escena?'); plt.grid(alpha=0.3); plt.show()
print('✅ Valores cercanos a 1 = la escena apenas cambia; caídas = cambio de plano o movimiento')

In [ ]:
# ============================================================
# CELDA 5.6 — Elegir fotogramas clave
# ============================================================
# Estrategia sencilla ("voraz"): recorremos los fotogramas y nos quedamos con uno solo si
# se parece MENOS de UMBRAL al último que elegimos. Así eliminamos los repetidos.

def elegir_clave(V, umbral=0.92, maximo=4):
    elegidos = [0]                                   # El primero siempre entra
    for i in range(1, len(V)):
        if float(V[i] @ V[elegidos[-1]]) < umbral:   # Suficientemente distinto al último elegido
            elegidos.append(i)
        if len(elegidos) == maximo:
            break
    if len(elegidos) == 1:                           # Vídeo casi estático: añadimos el último
        elegidos.append(len(V) - 1)
    return elegidos

claves = elegir_clave(V)

fig, ejes = plt.subplots(1, len(claves), figsize=(4 * len(claves), 3.5))
for eje, i in zip(np.atleast_1d(ejes), claves):
    eje.imshow(fotogramas[i]); eje.set_title(f'clave · {tiempos[i]:.1f} s'); eje.axis('off')
plt.show()
print(f'✅ {len(claves)} fotogramas clave → ≈{len(claves) * 256} tokens en lugar de {len(fotogramas) * 256}')

In [ ]:
# ============================================================
# CELDA 5.7 — Describir el vídeo: cada fotograma clave + resumen final
# ============================================================
# 1) gemma3 describe cada fotograma clave por separado (indicando su instante).
# 2) Una última llamada SOLO de texto une las descripciones en un relato del vídeo.

def describir_video(fotogramas, tiempos, claves):
    descripciones = []
    for i in claves:
        d, _ = preguntar_vlm('Describe en una frase en español lo que ocurre en este fotograma.',
                             [fotogramas[i]])
        descripciones.append(f'[{tiempos[i]:.1f} s] {d}')
    resumen, _ = preguntar_vlm(
        'Estas son descripciones de fotogramas de un mismo vídeo, en orden temporal. '
        'Escribe en español un resumen de 2-3 frases de lo que pasa en el vídeo:\n\n'
        + '\n'.join(descripciones))
    return descripciones, resumen

descripciones, resumen_video = describir_video(fotogramas, tiempos, claves)
print('\n'.join(descripciones))
print(f'\n🎬 Resumen del vídeo:\n{resumen_video}')
print('\n✅ PARTE 5 completa: vídeo → fotogramas clave → imágenes → texto')

## 🖥️ PARTE 6 — Interfaz Gradio multimodal

Juntamos todo en una app con tres pestañas:

| Pestaña | Entrada | Salida |
|---------|---------|--------|
| 🖼️ Imagen | Foto + pregunta | Cuadrícula de parches, respuesta de gemma3 y tokens que ocupó |
| 🔊 Audio | Archivo o micrófono | Transcripción de Whisper con tiempos + resumen |
| 🎬 Vídeo | Archivo de vídeo | Fotogramas clave + descripción del vídeo |

In [ ]:
# ============================================================
# CELDA 6.1 — Guardar los ejemplos como archivos para la interfaz
# ============================================================
# gr.Examples necesita rutas de archivo, así que guardamos las imágenes de skimage en disco.

os.makedirs('ejemplos', exist_ok=True)
rutas_imagenes = []
for nombre, imagen in IMAGENES.items():
    ruta = f'ejemplos/{nombre}.png'
    Image.fromarray(imagen).save(ruta)
    rutas_imagenes.append(ruta)

print('✅ Ejemplos guardados:', rutas_imagenes + [ruta_audio, ruta_video])

In [ ]:
# ============================================================
# CELDA 6.2 — Función de la pestaña Imagen
# ============================================================

def cuadricula_parches(imagen_np):
    """Devuelve la imagen 224×224 que ve CLIP, ampliada a 448 px, con la cuadrícula de parches."""
    px = procesador_clip(images=imagen_np, return_tensors='pt')['pixel_values']
    vis = Image.fromarray((a_imagen_visible(px) * 255).astype(np.uint8)).resize((448, 448))
    lienzo = np.array(vis)
    paso = 448 // N_LADO
    lienzo[::paso, :] = [255, 255, 0]   # Líneas horizontales amarillas cada 64 px
    lienzo[:, ::paso] = [255, 255, 0]   # Líneas verticales
    return lienzo

def ui_imagen(imagen, pregunta):
    if imagen is None:
        return None, '⚠️ Sube una imagen.', ''
    pregunta = pregunta.strip() or 'Describe esta imagen en español.'
    respuesta, tokens = preguntar_vlm(pregunta, [imagen])
    info = (f'🔢 **CLIP ViT-B/32:** {N_PARCHES} parches de {P}×{P} px + [CLS] = {N_PARCHES + 1} tokens  \n'
            f'🔢 **gemma3:** {tokens} tokens de entrada en total (imagen + pregunta)')
    return cuadricula_parches(imagen), respuesta, info

print('✅ ui_imagen definida')

In [ ]:
# ============================================================
# CELDA 6.3 — Funciones de las pestañas Audio y Vídeo
# ============================================================

def ui_audio(ruta):
    if ruta is None:
        return '⚠️ Sube un audio o grábalo con el micrófono.', ''
    texto, trozos = transcribir(ruta)
    lineas = [f'[{tr["timestamp"][0]:.1f}s] {tr["text"].strip()}' for tr in trozos]
    return '\n'.join(lineas) or texto, resumir_transcripcion(texto)

def ui_video(ruta, n_muestras, umbral):
    if ruta is None:
        return [], '⚠️ Sube un vídeo.'
    fgs, ts = extraer_fotogramas(ruta, n=int(n_muestras))
    cl = elegir_clave(vectores_clip(fgs), umbral=umbral)
    descs, resumen = describir_video(fgs, ts, cl)
    galeria = [(fgs[i], f'{ts[i]:.1f} s') for i in cl]   # Gallery acepta (imagen, pie de foto)
    return galeria, f'**Resumen:** {resumen}\n\n' + '\n\n'.join(descs)

print('✅ ui_audio y ui_video definidas')

In [ ]:
# ============================================================
# CELDA 6.4 — Construir la interfaz con gr.Blocks
# ============================================================
import gradio as gr   # Fijado a 5.x en la celda 1.4 (en Gradio 6, css y theme pasan a launch())

CSS = '#titulo {text-align: center}'

with gr.Blocks(title='Modelos Multimodales', css=CSS, theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎨🔊🎬 Modelos Multimodales\nCLIP · gemma3 · Whisper', elem_id='titulo')

    with gr.Tab('🖼️ Imagen'):
        with gr.Row():
            with gr.Column():
                in_img = gr.Image(type='numpy', label='Imagen')
                in_preg = gr.Textbox(label='Pregunta', placeholder='Describe esta imagen en español.')
                btn_img = gr.Button('Analizar', variant='primary')
                gr.Examples(rutas_imagenes, inputs=in_img)
            with gr.Column():
                out_grid = gr.Image(label='Así la trocea el tokenizador visual (7×7 parches)')
                out_info = gr.Markdown()
                out_resp = gr.Textbox(label='Respuesta de gemma3', lines=6)
        btn_img.click(ui_imagen, [in_img, in_preg], [out_grid, out_resp, out_info])

    with gr.Tab('🔊 Audio'):
        with gr.Row():
            with gr.Column():
                in_aud = gr.Audio(type='filepath', sources=['upload', 'microphone'], label='Audio')
                btn_aud = gr.Button('Transcribir', variant='primary')
                gr.Examples([ruta_audio], inputs=in_aud)
            with gr.Column():
                out_trans = gr.Textbox(label='Transcripción (Whisper)', lines=8)
                out_res = gr.Textbox(label='Resumen en español (gemma3)', lines=4)
        btn_aud.click(ui_audio, in_aud, [out_trans, out_res])

    with gr.Tab('🎬 Vídeo'):
        with gr.Row():
            with gr.Column():
                in_vid = gr.Video(label='Vídeo')
                in_n = gr.Slider(4, 24, value=12, step=1, label='Fotogramas muestreados')
                in_umb = gr.Slider(0.80, 0.99, value=0.92, step=0.01,
                                   label='Umbral de similitud (más alto = más fotogramas clave)')
                btn_vid = gr.Button('Describir vídeo', variant='primary')
                gr.Examples([ruta_video], inputs=in_vid)
            with gr.Column():
                out_gal = gr.Gallery(label='Fotogramas clave', columns=4, height=220)
                out_desc = gr.Markdown()
        btn_vid.click(ui_video, [in_vid, in_n, in_umb], [out_gal, out_desc])

print('✅ Interfaz construida (aún no lanzada)')

In [ ]:
# ============================================================
# CELDA 6.5 — Lanzar la interfaz dentro de Colab
# ============================================================
# Prueba cada pestaña con su ejemplo y luego con tus propios archivos
# (una foto tuya, una nota de voz en español, un vídeo corto del móvil).

demo.launch(quiet=True)

print('✅ Interfaz visible arriba 👆')

## 🤔 PARTE 7 — Reflexión

Responde en esta misma celda (doble clic para editarla). Entre 2 y 5 líneas por pregunta.

1. **Conceptual:** ¿En qué se parecen y en qué se diferencian un token de texto, un parche de imagen y un token de audio de Whisper? ¿Qué tienen en común cuando llegan al Transformer?

   *Tu respuesta:*

2. **Código:** En las celdas 2.7 y 2.8 comprobamos que la `Conv2d` del ViT da el mismo resultado que "aplanar el parche y multiplicarlo por W". ¿Por qué se usa una convolución con `kernel_size = stride = 32`? ¿Qué pasaría con el número de parches si el stride fuera 16?

   *Tu respuesta:*

3. **Implicaciones:** El preprocesado de CLIP recorta el centro de la imagen y Whisper solo escucha ventanas de 30 s. ¿Qué información se puede perder? Piensa en un caso real (médico, legal, accesibilidad) donde eso sería un problema.

   *Tu respuesta:*

4. **Comparativa:** Para "entender" un vídeo, ¿qué ventajas e inconvenientes tiene enviar fotogramas clave a un LLM multimodal frente a transcribir solo su audio con Whisper? ¿Cuándo usarías cada opción o ambas?

   *Tu respuesta:*

5. **Coste:** Según tus mediciones de la Parte 3 y la Parte 5, ¿cuántos tokens costaría analizar un vídeo de 1 minuto a 1 fotograma por segundo? ¿Cómo lo reducirías?

   *Tu respuesta:*

## 🏆 PARTE 8 — Retos opcionales

Cada reto tiene una celda vacía debajo. No están resueltos: solo una pista para empezar.

- **Reto A — Parches más pequeños:** carga `openai/clip-vit-base-patch16` y repite las celdas 2.4–2.6. ¿Cuántos parches salen ahora? ¿Cuántos tokens tendría la secuencia?
  *Pista:* cambia `NOMBRE_CLIP` y vuelve a ejecutar desde la celda 2.3; `cfg.patch_size` se actualiza solo.

- **Reto B — Whisper en español:** graba con el micrófono de la pestaña Audio una nota de voz en español y fuerza el idioma o pide la traducción al inglés.
  *Pista:* `transcriptor(ruta, generate_kwargs={'language': 'spanish'})` o `{'task': 'translate'}`.

- **Reto C — Vídeo con audio:** combina las Partes 4 y 5: extrae el audio de un vídeo propio, transcríbelo y pásale a gemma3 la transcripción junto con las descripciones de los fotogramas clave.
  *Pista:* `!ffmpeg -i video.mp4 -ar 16000 -ac 1 audio.wav` extrae el audio (ffmpeg ya está en Colab).

In [ ]:
# ============================================================
# CELDA 8.1 — Reto A: tu espacio para experimentar
# ============================================================
# (celda vacía a propósito)

In [ ]:
# ============================================================
# CELDA 8.2 — Reto B: tu espacio para experimentar
# ============================================================
# (celda vacía a propósito)

In [ ]:
# ============================================================
# CELDA 8.3 — Reto C: tu espacio para experimentar
# ============================================================
# (celda vacía a propósito)

## 🌐 PARTE 9 — Publicación con Ngrok

Hasta ahora la app solo vive dentro de Colab. Con **Ngrok** creamos un **túnel** hacia Internet
para que tus compañeros la abran desde su navegador (¡y prueben el micrófono desde el móvil!).

```
  Compañero (navegador)
        │  https://xxxx.ngrok-free.app
        ▼
  Servidor de Ngrok  ───túnel───►  Colab (localhost:7860)
                                          │
                                          ▼
                                   Tu app Gradio ──► CLIP · Whisper · gemma3
```

**Antes de empezar:** crea una cuenta gratuita en [ngrok.com](https://ngrok.com), copia tu *Authtoken*
y guárdalo en Colab: icono 🔑 **Secrets** → *Add new secret* → nombre `NGROK_TOKEN` → pega el token → activa *Notebook access*.

In [ ]:
# ============================================================
# CELDA 9.1 — Instalar pyngrok
# ============================================================
# Ya lo instalamos en la celda 1.4; repetirlo es instantáneo y garantiza que esté
# disponible aunque hayas reiniciado la sesión.

!pip install -q pyngrok

print('✅ pyngrok instalado')

In [ ]:
# ============================================================
# CELDA 9.2 — Autenticar con el token de Ngrok
# ============================================================
# El token es como una contraseña: NUNCA lo escribas directamente en el notebook
# si vas a compartirlo, porque cualquiera podría usar tu cuenta.
# Por eso lo leemos desde Colab Secrets.

from pyngrok import ngrok            # Librería para crear el túnel

try:
    from google.colab import userdata                 # Acceso a los Secrets de Colab
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')         # Lee el secreto llamado NGROK_TOKEN
except Exception as e:
    # Si Secrets no está disponible o el secreto no existe, usamos la variable manual.
    print(f'⚠️ No se pudo leer el Secret: {e}')
    print('⚠️ SEGURIDAD: si escribes el token aquí, bórralo antes de compartir el notebook.')
    NGROK_TOKEN = 'PEGA_TU_TOKEN_AQUI'                # Solo como último recurso

# Validamos que no quedó el texto de ejemplo
if NGROK_TOKEN == 'PEGA_TU_TOKEN_AQUI':
    raise ValueError('❌ Falta el token. Guárdalo en Secrets con el nombre NGROK_TOKEN y reintenta.')

ngrok.set_auth_token(NGROK_TOKEN)   # Registra el token para esta sesión

print('✅ Token de Ngrok configurado')

In [ ]:
# ============================================================
# CELDA 9.3 — Abrir el túnel y relanzar la app
# ============================================================
# Orden importante: cerrar lo anterior → abrir túnel → lanzar la app.

demo.close()    # Cierra la app lanzada en la Parte 6 para liberar el puerto
ngrok.kill()    # Cierra túneles anteriores (el plan gratuito solo permite uno a la vez)

# connect(7860) crea un túnel público hacia el puerto local 7860
tunel = ngrok.connect(7860)
print(f'🌍 URL pública: {tunel.public_url}')
print('   Compártela con tus compañeros y ábrela en tu celular para comprobarla.')

# server_name='0.0.0.0' = acepta conexiones externas (las que llegan por el túnel)
# server_port=7860       = el mismo puerto que expusimos con Ngrok
# share=False            = no usamos el enlace temporal de Gradio; Ngrok es más estable
demo.launch(server_name='0.0.0.0', server_port=7860, share=False, quiet=True)

print('✅ App publicada con Ngrok')

In [ ]:
# ============================================================
# CELDA 9.4 — Verificar los túneles activos
# ============================================================
# get_tunnels() lista los túneles abiertos en esta sesión.

tuneles = ngrok.get_tunnels()

if tuneles:
    print(f'✅ {len(tuneles)} túnel(es) activo(s):')
    for t in tuneles:
        print(f'   {t.public_url}  →  {t.config["addr"]}')
else:
    print('❌ No hay túneles activos. Ejecuta de nuevo la celda 9.3.')

In [ ]:
# ============================================================
# CELDA 9.5 — (Opcional) Dominio estático
# ============================================================
# Con el plan gratuito puedes reclamar UN dominio fijo en dashboard.ngrok.com → Domains.
# Así la URL no cambia entre sesiones. Si no tienes uno, sáltate esta celda.

DOMINIO_ESTATICO = ''   # Ej: 'mi-app-multimodal.ngrok-free.app'

if DOMINIO_ESTATICO:
    ngrok.kill()                                                    # Cierra el túnel aleatorio
    tunel = ngrok.connect(7860, domain=DOMINIO_ESTATICO)            # Abre el túnel con tu dominio
    print(f'✅ URL fija: {tunel.public_url}')
else:
    print('ℹ️ Sin dominio estático configurado: se mantiene la URL aleatoria.')

In [ ]:
# ============================================================
# CELDA LIMPIEZA — Liberar todos los recursos al terminar
# ============================================================
# Orden: túnel → interfaz → procesos → memoria GPU.

# 1. Cerrar el túnel Ngrok
ngrok.kill()
print('✅ Túnel Ngrok cerrado')

# 2. Cerrar la interfaz Gradio
demo.close()
print('✅ Interfaz Gradio cerrada')

# 3. Detener el servidor Ollama (libera también la memoria de gemma3)
proceso_ollama.terminate()
print('✅ Servidor Ollama detenido')

# 4. Liberar la memoria GPU de CLIP y Whisper
del modelo_clip, modelo_whisper, transcriptor   # Borra las referencias a los modelos
torch.cuda.empty_cache()                        # Devuelve a la GPU la memoria reservada por PyTorch
print('✅ Memoria GPU liberada')

print('\n✅ Todos los recursos liberados. Puedes cerrar el notebook.')

## 📊 Rúbrica de Evaluación

| Criterio | Excelente (5) | Satisfactorio (3) | En desarrollo (1) |
|----------|---------------|-------------------|-------------------|
| **Preparación del entorno** | Parte 1 completa sin errores: GPU, librerías de Colab y `gemma3:4b` listos | Funciona tras algún reintento o reinicio | No logra cargar los modelos |
| **Tokenizador visual** | Explica los 4 pasos (preprocesar, parches, vector, posición) y la equivalencia Conv2d = matriz | Ejecuta la Parte 2 pero explica solo parte del proceso | No identifica qué es un parche |
| **Audio con Whisper** | Interpreta el espectrograma, los 1500 tokens de audio y los tokens especiales del decoder | Obtiene la transcripción sin interpretar los pasos | La transcripción falla |
| **Vídeo y coste en tokens** | Calcula el coste en tokens y justifica la selección de fotogramas clave | Obtiene la descripción del vídeo sin analizar el coste | No completa la Parte 5 |
| **Publicación con Ngrok** | URL pública funcionando y abierta desde otro dispositivo | Ngrok conecta pero sin verificar desde otro dispositivo | No intenta la publicación |
| **Reflexión y retos** | 5 respuestas razonadas e intenta al menos un reto | Respuestas incompletas o genéricas | Sin respuestas |